# Импорты

In [ ]:
import os
import random
import cv2
import timm
import copy

import pandas as pd
import seaborn as sns
import albumentations as A
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from collections import defaultdict, Counter
from glob import glob
from PIL import Image

# Сиды

In [ ]:
# важно - зафиксировать все сиды
SEED = 8888
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Классы

In [ ]:
class_to_idx = { "customer": 0,
                 "staff": 1 }
classes = ['customer', 'staff']

# Инкапсуляция

In [ ]:
class MyDataset(Dataset):
    def __init__(self, root_dir, class_to_idx, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        self.images = []
        self.labels = []

        for class_name, label_id in self.class_to_idx.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                print(f"Папка {class_dir} не найдена!")
                continue

            for file_name in os.listdir(class_dir):
                file_path = os.path.join(class_dir, file_name)
                self.images.append(file_path)
                self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        image = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.labels[idx]
        if self.transform:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented['image']

        return image, label


In [ ]:
dataset_path = '/content/dataset'

# Анализ данных

In [ ]:
class_counts = defaultdict(int)
widths = []
heights = []
areas = []

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)
    if not os.path.isdir(class_dir):
        print(f"Папка {class_dir} не найдена!")
        continue

    for fname in os.listdir(class_dir):
        fpath = os.path.join(class_dir, fname)
        if fname.lower().endswith(('.jpg')):
            try:
                with Image.open(fpath) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    areas.append(w * h)
                    class_counts[class_name] += 1
            except Exception as e:
                print(f"Ошибка при обработке {fpath}: {e}")

print("Количество изображений по классам:")
for cls in classes:
    print(f"{cls}: {class_counts[cls]}")

areas = np.array(areas)
widths = np.array(widths)
heights = np.array(heights)

quantile_90_area = np.percentile(areas, 90)
print(f"\n90-й квантиль площади: {quantile_90_area:.2f} пикселей²")

# Графики
fig, axes = plt.subplots(2, 1, figsize=(8, 8))

sns.scatterplot(x=widths, y=heights, alpha=0.3, ax=axes[0], color='blue')
axes[0].set_title('Ширина vs Высота')
axes[0].set_xlabel('Ширина')
axes[0].set_ylabel('Высота')
axes[0].scatter([300], [300], color='red', s=100, label='Model Input (300x300)', zorder=5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

axes[1].hist(areas, bins=30, color='lightgreen', edgecolor='black')
axes[1].axvline(quantile_90_area, color='red', linestyle='dashed', linewidth=2, label=f'90% = {quantile_90_area:.0f}')
axes[1].set_title('Распределение площади')
axes[1].set_xlabel('Площадь (пиксели²)')
axes[1].set_ylabel('Частота')
axes[1].legend()

plt.tight_layout()
plt.show()

# DEVICE

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Аугументация

In [ ]:
IMG_SIZE = 224

train_transforms = A.Compose([
    A.RandomResizedCrop(
        size=(IMG_SIZE,IMG_SIZE),
        scale=(0.65, 1.0),       # доля площади исходного патча
        ratio=(0.95, 1.05),       # допустимое изменение пропорций (почти не искажает)
        p=0.2
    ),

    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(
        min_height=IMG_SIZE,
        min_width=IMG_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=1.0
    ),

    A.ShiftScaleRotate(
        shift_limit=0.2,
        scale_limit=0.1,
        rotate_limit=5,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.75
    ),

    A.HorizontalFlip(p=0.5),

    A.OneOf([
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        A.CLAHE(p=0.1),
    ], p=0.85),

    # Освещение и цвет
    A.OneOf([
        A.ColorJitter(brightness=0.1, contrast=0.0, saturation=0.0, hue=0.0),
        A.ColorJitter(brightness=0.0, contrast=0.1, saturation=0.0, hue=0.0),
        A.ColorJitter(brightness=0.0, contrast=0.0, saturation=0.1, hue=0.0),
    ], p=1),

    # Искажения
    A.OneOf([
        A.GaussianBlur(blur_limit=(3), sigma_limit=(0.1, 1.0), p=0.5),
        A.GaussNoise(std_range=(0.01, 0.02), noise_scale_factor=0.7, p=0.9),
    ], p=0.9),

    A.CoarseDropout(
        num_holes_range=(4, 16),
        hole_height_range=(0.02, 0.1),
        hole_width_range=(0.02, 0.1),
        fill=0,
        p=0.75
    ),

    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    A.ToTensorV2(),
])

val_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(
        min_height=IMG_SIZE,
        min_width=IMG_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=1.0
    ),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
def visualize_augmentations(dataset, idx=None, n_samples=6):
    if idx is None:
        idx = random.randint(0, len(dataset) - 1)

    # Достаем путь к файлу из нашего списка путей
    image_path = dataset.images[idx]

    # Читаем оригинал
    image = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    viz_transforms = A.Compose([
        t for t in train_transforms.transforms
        if not isinstance(t, (A.Normalize, A.pytorch.ToTensorV2))
    ])

    # 3. Рисуем сетку
    plt.figure(figsize=(15, 10))

    # Показываем оригинал первым
    plt.subplot(2, 3, 1)
    plt.imshow(image)
    plt.title("Оригинал")
    plt.axis('off')

    # Применяем аугментацию n-1 раз
    for i in range(2, n_samples + 1):
        augmented = viz_transforms(image=image)["image"]

        plt.subplot(2, 3, i)
        plt.imshow(augmented)
        plt.title(f"Вариант {i-1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# --- ЗАПУСК ---
# Передай свой train_dataset в функцию
dataset = MyDataset(dataset_path, class_to_idx=class_to_idx, transform=train_transforms)
visualize_augmentations(dataset)

# Валидация


In [ ]:
@torch.no_grad() # при вызове функции отключаем калькулятор градиентов
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # Списки для накопления всех ответов
    all_preds = []
    all_targets = []

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for X_batch, y_batch in pbar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == y_batch).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)

        # Обновляем прогресс-бар (показываем acc, так как f1 на лету не посчитать)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

        # Сохраняем предсказания и реальные метки в список
        all_preds.extend(y_pred.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

    # Финальные метрики
    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)

    # Считаем F1 по всему датасету разом
    f1 = f1_score(all_targets, all_preds, average='binary')

    return accuracy, avg_loss, f1

# Функция train

In [ ]:
def train(model, loss_fn, optimizer, train_loader, val_loader, device,\
          writer=None, n_epoch=3, lr_eta=1e-6, ls=0.05):
    best_val_f1 = 0.0
    num_iter = 0
    counter_early_stop = 0
    save_path = f"checkpoints/best_model.pth"
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epoch, eta_min=lr_eta) # планировщик скорости обучения
    loss_fn_val = torch.nn.CrossEntropyLoss(label_smoothing=ls)
    mixup_fn = Mixup(
        mixup_alpha=0.2,
        cutmix_alpha=0.0,
        prob=0.25,
        label_smoothing=0.05,
        num_classes=2
    )

    for epoch in range(1, n_epoch + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Ep {epoch}/{n_epoch}", leave=True, dynamic_ncols=True)

        for i, (X_batch, y_batch) in enumerate(pbar):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            X_batch, y_batch = mixup_fn(X_batch, y_batch)

            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            # накопим метрики для прогресс-бара
            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_true_cls = y_batch.argmax(dim=1)
            y_pred_cls = logits.argmax(dim=1)

            correct = (y_pred_cls == y_true_cls).sum().item()
            total_correct += correct

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # Обновляем tqdm: показываем Loss, Acc и текущий LR
            current_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({
                'loss': f"{avg_loss:.4f}",
                'acc': f"{acc:.4f}",
                'lr': f"{current_lr:.6f}"
            })

            # логирование
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", correct / batch_size, num_iter)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        # Валидация (тоже с tqdm)
        val_acc, val_loss, val_f1 = evaluate(model, val_loader, loss_fn_val, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)
            writer.add_scalar(f"F1/val", val_f1, num_iter)
            writer.add_scalar(f"LR", current_lr, epoch)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        # Проверка на лучшую модель
        if val_f1 >= best_val_f1:
            best_val_f1 = val_f1
            counter_early_stop = 0
            torch.save(model.state_dict(), save_path)
            print(f"--- Эпоха {epoch}: Новый лучший F1: {val_f1:.4f}! Модель сохранена. ---")
        else:
            counter_early_stop += 1
            if counter_early_stop >= 5:
                print(f"Сработал Early stop! Лучший F1 был: {best_val_f1:.4f}")
                break

    # Загружаем лучший вес
    model.load_state_dict(torch.load(f"checkpoints/best_model.pth"))
    os.remove(f"checkpoints/best_model.pth")
    return model, best_val_f1

# Итоговая метрика

In [ ]:
@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device, non_blocking=True)

        logits = model(X_batch)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(y_batch.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # names for report
    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

# Параметры обучения

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
writer = SummaryWriter("logs")

n_fold = 5
ls = 0.05
skf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
loss_fn = SoftTargetCrossEntropy()

epoch = 15
b_size = 32
lr_ = 1e-4
lr_eta = 1e-6
model_name = 'convnext_tiny.fb_in22k_ft_in1k'
data_config = timm.data.resolve_model_data_config(timm.create_model(model_name, pretrained=True))
print(data_config)

In [ ]:
dataset_val = copy.deepcopy(dataset)
dataset_val.transforms = val_transforms

# Обучение

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
labels = dataset.labels

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(dataset, labels)):
    if fold_idx not in FOLDS_TO_TRAIN:
        continue
    print(f"\n--- Запуск фолда {fold_idx + 1}/{n_fold}  ---")

    # Выбираем файлы по индексам, которые дал SKF
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset_val, val_idx)

    # Семплирование
    train_labels = [labels[i] for i in train_idx]
    class_counts = np.bincount(train_labels)
    class_weights_sampler = 1.0 / class_counts
    sample_weights = [class_weights_sampler[label] for label in train_labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_subset, batch_size=b_size, sampler=sampler, num_workers=2, pin_memory=True, persistent_workers=True, drop_last=True)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=2,
        drop_rate=0.4,
        drop_path_rate=0.1
    )
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_, weight_decay=1e-2)

    model, f1 = train(model, loss_fn, optimizer, train_loader, val_loader,
                  device, writer=None, n_epoch=epoch, lr_eta=lr_eta)

    sklearn_report(model, val_loader, device, idx2class={v: k for k, v in class_to_idx.items()})

    # Сохраняем модель текущего фолда
    save_path = f"checkpoints/{model_name}_{fold_idx+1}_{f1}.pth"
    torch.save(model.state_dict(), save_path)

    # Очистка памяти GPU перед следующим фолдом
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()